In [86]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle

In [92]:
import shap, pickle
with open('trained_models2_pkl/trained_models2_34_feat_XGB_fly.pkl', 'rb') as f:
    best_models = pickle.load(f)

print(best_models.keys())

dict_keys(['Mean Fluorescence Intensity', 'Cell viability', 'Transfection efficiency'])


In [93]:
df = pd.read_excel("34_feat_dataset.xlsx")
# replicate your original preprocessing
from sklearn.preprocessing import StandardScaler

X = df.iloc[:, :-5]

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

In [94]:
def compute_shap_importance_xgb(model, X):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    return np.abs(shap_values).mean(axis=0)

importance_dict = {}

for output_name, model in best_models.items():
    shap_imp = compute_shap_importance_xgb(model, X_scaled)
    importance_dict[output_name] = shap_imp

importance_df = pd.DataFrame(importance_dict, index=X_scaled.columns)


In [95]:
def categorize(val, q1, q2, q3):
    if val >= q3:
        return "HIGH"
    elif val >= q2:
        return "MEDIUM"
    elif val >= q1:
        return "LOW"
    else:
        return "POOR"

output_tables = {}

for col in importance_df.columns:
    
    df_out = pd.DataFrame({
        "Feature": importance_df.index,
        "SHAP Importance": importance_df[col].values
    })
    
    # sort by importance
    df_out = df_out.sort_values("SHAP Importance", ascending=False)
    
    # quartiles (per output)
    q1 = df_out["SHAP Importance"].quantile(0.25)
    q2 = df_out["SHAP Importance"].quantile(0.50)
    q3 = df_out["SHAP Importance"].quantile(0.75)
    
    # categorize
    df_out["Category"] = df_out["SHAP Importance"].apply(
        lambda x: categorize(x, q1, q2, q3)
    )
    
    # keep top 25
    output_tables[col] = df_out.head(25)

In [96]:
for name, table in output_tables.items():
    print("\n", name)
    print(table)
    #table.to_excel(f"{name}_bump.plot.top20_features.xlsx", index=False)


 Mean Fluorescence Intensity
          Feature  SHAP Importance Category
8      cargo_conc         0.417568     HIGH
3      NS_density         0.342693     HIGH
6       cargo_CAR         0.287552     HIGH
0         voltage         0.131928     HIGH
13  NS_depth_cyto         0.127775     HIGH
4    NS_centforce         0.093836     HIGH
1     pulse_count         0.087190     HIGH
14       NS_count         0.077075     HIGH
2      osmolarity         0.071821     HIGH
12   NS_depth_nuc         0.063626   MEDIUM
10      exp_NKG2D         0.053613   MEDIUM
19        exp_Prf         0.050732   MEDIUM
11       exp_Ki67         0.046259   MEDIUM
33       cell_CD8         0.042353   MEDIUM
7       cargo_miR         0.034599   MEDIUM
23     exp_CD107a         0.029069   MEDIUM
9        act_days         0.028801   MEDIUM
5       NS_height         0.023088      LOW
32      cell_HeLa         0.009637      LOW
22       exp_TNFa         0.006208      LOW
18       exp_TIM3         0.005755      LOW
24

In [14]:
output_tables["Transfection efficiency"]
output_tables["Cell viability"]
output_tables["Mean Fluorescence Intensity"]

,Feature,SHAP Importance,Category
8,cargo_conc,0.417568,HIGH
3,NS_density,0.342693,HIGH
6,cargo_CAR,0.287552,HIGH
0,voltage,0.131928,HIGH
13,NS_depth_cyto,0.127775,HIGH
4,NS_centforce,0.093836,HIGH
1,pulse_count,0.087190,HIGH
14,NS_count,0.077075,HIGH
2,osmolarity,0.071821,HIGH
12,NS_depth_nuc,0.063626,MEDIUM


### FUNCTION RESULTS exports xlsx OF TOP 25 FEATURES with FEATURE VALUE

### USED TO PLOT BUMP PLOTS FOR OUTPUT (XGB)

In [98]:
def generate_feature_tables(
    model_name="XGB",
    datafile="34_feat_dataset.xlsx",
    pkl_path="trained_models2_pkl/",
    top_n=25,
    save=False
):
    """
    Generate SHAP-based feature importance tables per output.

    Parameters:
    -----------
    model_name : str
        Model identifier (e.g., "XGB")
    datafile : str
        Path to dataset
    pkl_path : str
        Folder containing trained model pickle files
    top_n : int
        Number of top features to keep per output
    save : bool
        Whether to save tables as Excel files

    Returns:
    --------
    output_tables : dict
        Dictionary of DataFrames (one per output)
    importance_df : pd.DataFrame
        Raw SHAP importance values
    """

    import pandas as pd
    import numpy as np
    import shap
    import pickle
    from sklearn.preprocessing import StandardScaler

    # Load models
    pkl_file = f"{pkl_path}trained_models2_34_feat_{model_name}_fly.pkl"
    with open(pkl_file, 'rb') as f:
        best_models = pickle.load(f)

    print("Loaded models:", best_models.keys())

    # Load dataset
    df = pd.read_excel(datafile)

    # Recreate training preprocessing
    X = df.iloc[:, :-5]

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    # SHAP importance function
    def compute_shap_importance(model, X):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        return np.abs(shap_values).mean(axis=0)

    # Compute importance per output
    importance_dict = {}

    for output_name, model in best_models.items():
        shap_imp = compute_shap_importance(model, X_scaled)
        importance_dict[output_name] = shap_imp

    importance_df = pd.DataFrame(importance_dict, index=X_scaled.columns)

    # Categorization function
    def categorize(val, q1, q2, q3):
        if val >= q3:
            return "HIGH"
        elif val >= q2:
            return "MEDIUM"
        elif val >= q1:
            return "LOW"
        else:
            return "POOR"

    # Build output tables
    output_tables = {}

    for col in importance_df.columns:

        df_out = pd.DataFrame({
            "Feature": importance_df.index,
            "SHAP Importance": importance_df[col].values
        })

        df_out = df_out.sort_values("SHAP Importance", ascending=False)

        # quartiles (per output)
        q1 = df_out["SHAP Importance"].quantile(0.25)
        q2 = df_out["SHAP Importance"].quantile(0.50)
        q3 = df_out["SHAP Importance"].quantile(0.75)

        df_out["Category"] = df_out["SHAP Importance"].apply(
            lambda x: categorize(x, q1, q2, q3)
        )

        df_out = df_out.head(top_n)

        output_tables[col] = df_out

        #Save if needed
        if save:
            safe_name = col.replace(" ", "_")
            df_out.to_excel(f"{safe_name}_{model_name}_top{top_n}_bumpplot.xlsx", index=False)

    return output_tables, importance_df

In [100]:
tables, importance = generate_feature_tables(
    model_name="XGB",
    top_n=25,
    save=True
)

Loaded models: dict_keys(['Mean Fluorescence Intensity', 'Cell viability', 'Transfection efficiency'])


In [101]:
tables["Cell viability"]

,Feature,SHAP Importance,Category
33,cell_CD8,3.033887,HIGH
9,act_days,1.480306,HIGH
32,cell_HeLa,1.234425,HIGH
0,voltage,1.214350,HIGH
1,pulse_count,0.911098,HIGH
5,NS_height,0.777957,HIGH
13,NS_depth_cyto,0.773679,HIGH
4,NS_centforce,0.709988,HIGH
7,cargo_miR,0.676329,HIGH
12,NS_depth_nuc,0.584490,MEDIUM


### FUNCTION RESULTS IN EXCELS OF TOP 25 FEATURES SEGREGATED INTO FAETURE VALUE


### USED TO PLOT BUMP PLOTS FOR ----CELL TYPE---- (XGB)

In [106]:
import pandas as pd
import numpy as np
import shap
import pickle
from sklearn.preprocessing import StandardScaler

# load model
with open('trained_models2_pkl/trained_models2_34_feat_XGB_fly.pkl', 'rb') as f:
    best_models = pickle.load(f)

# load data
df = pd.read_excel("34_feat_dataset.xlsx")

# features + cell type
X = df.iloc[:, :-5]
cell_types = df.iloc[:, -4].astype(str).str.strip()

# scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

results = {}

for output_name, model in best_models.items():

    print(f"\nProcessing output: {output_name}")

    # SHAP values mean
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_scaled)
    shap_abs = np.abs(shap_values)

    shap_df = pd.DataFrame(shap_abs, columns=X_scaled.columns)
    shap_df["cell_type"] = cell_types

    celltype_tables = {}
    for ct in shap_df["cell_type"].unique():

        subset = shap_df[shap_df["cell_type"] == ct].drop(columns="cell_type")

        # mean SHAP value
        mean_shap = subset.mean(axis=0)

        df_out = pd.DataFrame({
            "Feature": mean_shap.index,
            "SHAP Importance": mean_shap.values
        }).sort_values("SHAP Importance", ascending=False)

        # quartiles calculation
        q1 = df_out["SHAP Importance"].quantile(0.25)
        q2 = df_out["SHAP Importance"].quantile(0.50)
        q3 = df_out["SHAP Importance"].quantile(0.75)

        # categorize values
        def categorize(x):
            if x >= q3:
                return "HIGH"
            elif x >= q2:
                return "MEDIUM"
            elif x >= q1:
                return "LOW"
            else:
                return "POOR"

        df_out["Category"] = df_out["SHAP Importance"].apply(categorize)
        celltype_tables[ct] = df_out.head(25)

    results[output_name] = celltype_tables
    with pd.ExcelWriter("SHAP_results.xlsx", engine="xlsxwriter") as writer:
        for output_name, ct_dict in results.items():
            for cell_type, df_out in ct_dict.items():

                sheet_name = f"{output_name[:15]}_{cell_type[:15]}"
                df_out.to_excel(writer, sheet_name=sheet_name, index=False)

    print("SHAP results saved to SHAP_results.xlsx")



Processing output: Mean Fluorescence Intensity
SHAP results saved to SHAP_results.xlsx

Processing output: Cell viability
SHAP results saved to SHAP_results.xlsx

Processing output: Transfection efficiency
SHAP results saved to SHAP_results.xlsx


In [105]:
pip install XlsxWriter

Note: you may need to restart the kernel to use updated packages.


In [84]:
results["Transfection efficiency"]["NK"]

,Feature,SHAP Importance,Category
6,cargo_CAR,14.501228,HIGH
8,cargo_conc,8.656135,HIGH
26,cargo_GFP,5.266836,HIGH
0,voltage,2.162961,HIGH
3,NS_density,2.132667,HIGH
5,NS_height,2.122625,HIGH
1,pulse_count,1.316666,HIGH
9,act_days,1.252714,HIGH
12,NS_depth_nuc,1.121607,HIGH
13,NS_depth_cyto,1.055651,MEDIUM
